# 0. Exportamos paquetes y archivos
---

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
import janitor
import os
from pathlib import Path
import openpyxl
import sqlalchemy as sa

# Desactivar notación científica
pd.set_option('display.float_format', lambda x: '%.3f' % x)
np.set_printoptions(suppress=True)

# Cargar variables de entorno
load_dotenv()

print("✅ Librerías importadas correctamente")

✅ Librerías importadas correctamente


In [3]:
# Ruta de los archivos

ruta_proyecto = "../02_datos/03_Entrenamiento/"
ruta_cat = ruta_proyecto + "03_cat_resultado_eda.pkl"
ruta_num = ruta_proyecto + "03_num_resultado_eda.pkl"

In [4]:
# Cargamos los datos

cat = pd.read_pickle(ruta_cat)
num = pd.read_pickle(ruta_num)

print("✅ Dataframe 'cat' exportado correctamente")
print("✅ Dataframe 'num' exportado correctamente")

✅ Dataframe 'cat' exportado correctamente
✅ Dataframe 'num' exportado correctamente


In [5]:
# Separamos target 

target = num[['compra']].copy().reset_index(drop=True)
target

,compra
0,0
1,0
2,1
3,0
4,0
...,...
4832,1
4833,1
4834,0
4835,1


# 01. Transformación de categóricas
---

Vamos a analizar qué variables debemos transformar, en concreto con algún tipo de **Encoding**, por ejemplo, un Target Encoding podría aplicar a todas pero en ese caso si usamos un One-hot-encoding y luego hacemos target encoding, tendríamos muchas variables, con el problema de dimensionalidad que eso supone.
- `origen`: como no hay ningún orden (ordinal), aplicamos one-hot-encoding.
- `fuente`: igual que el anterior, aplicamos one-hot-encoding.
- `ult_actividad`: aplicamos one-hot-encoding.
- `ambito`: aplicamos one-hot-encoding.
- `ocupacion`: aquí se podría pensar que un *Working Professional* tendría un mayor poder adquisitivo que un *Student* pero no está claro, ya que no siempre podría ser así. Si hay dudas, se deja one-hot-encoding.
- `descarga_lm`: aplicamos one-hot-encoding.

![alt text](<Captura de pantalla 2026-06-15 a las 13.59.53.png>)

## 1.1. One Hot Encoding

In [6]:
var_ohe = ['origen', 'fuente', 'ult_actividad', 'ambito', 'ocupacion', 'descarga_lm']

In [ ]:
from sklearn.preprocessing import OneHotEncoder

# Instanciamos OHE
ohe = OneHotEncoder(sparse_output = False, handle_unknown = 'ignore')

# Entrenamos y aplicar
cat_ohe = ohe.fit_transform(cat[var_ohe])

# Guardamos como dataframe
cat_ohe = pd.DataFrame(cat_ohe, columns=ohe.get_feature_names_out())

## 1.2. Unificamos dataframes

In [8]:
pd.concat([cat_ohe, num], axis=1)

,origen_API,origen_Landing Page Submission,origen_Lead Add Form,origen_OTROS,fuente_Chat,fuente_Direct Traffic,fuente_Google,fuente_OTROS,fuente_Organic Search,fuente_Reference,...,ocupacion_Unemployed,ocupacion_Working Professional,descarga_lm_No,descarga_lm_Yes,compra,visitas_total,tiempo_en_site_total,paginas_vistas_visita,score_actividad,score_perfil
0,0.000,1.000,0.000,0.000,0.000,0.000,1.000,0.000,0.000,0.000,...,1.000,0.000,1.000,0.000,NaN,<NA>,NaN,NaN,NaN,NaN
1,1.000,0.000,0.000,0.000,1.000,0.000,0.000,0.000,0.000,0.000,...,1.000,0.000,1.000,0.000,NaN,<NA>,NaN,NaN,NaN,NaN
2,0.000,1.000,0.000,0.000,0.000,0.000,1.000,0.000,0.000,0.000,...,1.000,0.000,1.000,0.000,NaN,<NA>,NaN,NaN,NaN,NaN
3,1.000,0.000,0.000,0.000,1.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,1.000,0.000,NaN,<NA>,NaN,NaN,NaN,NaN
4,0.000,1.000,0.000,0.000,0.000,0.000,1.000,0.000,0.000,0.000,...,1.000,0.000,1.000,0.000,NaN,<NA>,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
595392,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,1.000,3,521.000,3.000,14.000,16.000
652250,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,1.000,3,981.000,3.000,14.000,15.000
606524,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.000,1,517.000,1.000,14.000,16.000
651199,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,1.000,5,893.000,5.000,15.000,15.000


Observamos un problema de columnas, ya que vemos muchos NaN en los id finales. Es un **problema de registros** ya que nos dan 9650 cuando teníamos 4225 porque no es capaz de identificar a qué registros de `cat` con `num` por el id, es decir, los duplica porque no tiene `id` algunas de ellas, en este caso sería la de `cat_ohe`. Tenemos dos opciones, **pegarle el `id` al `cat_ohe`** o **eliminarlo a `num`** y ya genera un secuencial que si funcionaría.

In [12]:
df = pd.concat([cat_ohe, num.reset_index()], axis=1)

# 02. Transformación de numéricas
---

La variable `compra`: es la *target* por lo que **no se realiza transformación**. 

Analizando qué variables vamos a transformar, se analiza el tipo de transformación:

- **Discretización**: proporciona mayor explicabilidad en proyectos con ese objetivo
- **Binarización**: No se realizará.
- **Normalización**: cuando se hizo el EDA en numéricas se veían "normalizadas" las variables.
- **Reescalado**: como vamos a hacer un **K-Means** y es muy sensible a la escala. Vamos a aplicar un *Robust Scaling* se descarta porque no tenemos problema de atípicos, entre *Min-Max Scaling* y *Standard Scaling*. Al tener varias variables que vamos a aplicar one-hot-encoding, es decir, tendremos variables en rangos de 0-1 tiene más sentido aplicar **Min-Max Scaling** que lo lleva a 0-1. Si hiciésamos, modelización predictiva, no importaría que estuviesen en distintas escalas, pero al hacer un Clúster, es importante que estén en la misma escala.

Para todas las variables numéricas, se usará Min-Max Scaling:  `visitas_total`, `tiempo_en_site_total`, `paginas_vistas_visita`, `score_actividad`, `score_perfil`.

## 2.1. Min-Max Scaling
Se aplicaría a todas las columnas a partir de las 39 (`visita_total`)

In [44]:
var_mms = df.iloc[:,39:].columns

In [45]:
from sklearn.preprocessing import MinMaxScaler

# Instanciamos 
mms = MinMaxScaler()

# Entrenamos y aplicar
df_mms = mms.fit_transform(df[var_mms])

# Añadimos sujijos a los nombres

nombres_mms = [variable + '_mms' for variable in var_mms]

# Guardamos en dataframe
df_mms = pd.DataFrame(df_mms, columns=nombres_mms)

# 03. Unificamos datasets de variables transformadas
---

In [46]:
# Investigamos como quedaría así

pd.concat([cat_ohe, df_mms], axis = 1).info()

<class 'pandas.DataFrame'>
RangeIndex: 4837 entries, 0 to 4836
Data columns (total 42 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   origen_API                                4837 non-null   float64
 1   origen_Landing Page Submission            4837 non-null   float64
 2   origen_Lead Add Form                      4837 non-null   float64
 3   origen_OTROS                              4837 non-null   float64
 4   fuente_Chat                               4837 non-null   float64
 5   fuente_Direct Traffic                     4837 non-null   float64
 6   fuente_Google                             4837 non-null   float64
 7   fuente_OTROS                              4837 non-null   float64
 8   fuente_Organic Search                     4837 non-null   float64
 9   fuente_Reference                          4837 non-null   float64
 10  ult_actividad_Chat Conversation           4837 

Vemos que la **variable `id` se ha perdido**, de forma que debemos incorporarla desde el dataset `df` y también la **variable target**.

Además, confirmamos que **no tenemos nulos**.

In [48]:
pd.concat([df.id, cat_ohe, df_mms, target], axis=1).isna().sum()

id                                          0
origen_API                                  0
origen_Landing Page Submission              0
origen_Lead Add Form                        0
origen_OTROS                                0
fuente_Chat                                 0
fuente_Direct Traffic                       0
fuente_Google                               0
fuente_OTROS                                0
fuente_Organic Search                       0
fuente_Reference                            0
ult_actividad_Chat Conversation             0
ult_actividad_Converted to Lead             0
ult_actividad_Email Link Clicked            0
ult_actividad_Email Opened                  0
ult_actividad_OTROS                         0
ult_actividad_Page Visited on Website       0
ult_actividad_SMS Sent                      0
ambito_Banking, Investment And Insurance    0
ambito_Business Administration              0
ambito_Finance Management                   0
ambito_Healthcare Management      

In [52]:
# Revisamos que todas las variables están en rangos de 0-1.

pd.concat([df.id, cat_ohe, df_mms, target], axis=1).set_index('id').describe().T

,count,mean,std,min,25%,50%,75%,max
origen_API,4837.000,0.314,0.464,0.000,0.000,0.000,1.000,1.000
origen_Landing Page Submission,4837.000,0.631,0.483,0.000,0.000,1.000,1.000,1.000
origen_Lead Add Form,4837.000,0.048,0.215,0.000,0.000,0.000,0.000,1.000
origen_OTROS,4837.000,0.007,0.084,0.000,0.000,0.000,0.000,1.000
fuente_Chat,4837.000,0.060,0.238,0.000,0.000,0.000,0.000,1.000
fuente_Direct Traffic,4837.000,0.323,0.467,0.000,0.000,0.000,1.000,1.000
fuente_Google,4837.000,0.392,0.488,0.000,0.000,0.000,1.000,1.000
fuente_OTROS,4837.000,0.030,0.170,0.000,0.000,0.000,0.000,1.000
fuente_Organic Search,4837.000,0.151,0.358,0.000,0.000,0.000,0.000,1.000
fuente_Reference,4837.000,0.045,0.207,0.000,0.000,0.000,0.000,1.000


# 04. Creamos el dataframe final
---

In [ ]:
df_tablon = pd.concat([df.id, cat_ohe, df_mms, target], axis=1).set_index('id')

In [55]:
df_tablon.info()

<class 'pandas.DataFrame'>
Index: 4837 entries, 630952 to 592736
Data columns (total 43 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   origen_API                                4837 non-null   float64
 1   origen_Landing Page Submission            4837 non-null   float64
 2   origen_Lead Add Form                      4837 non-null   float64
 3   origen_OTROS                              4837 non-null   float64
 4   fuente_Chat                               4837 non-null   float64
 5   fuente_Direct Traffic                     4837 non-null   float64
 6   fuente_Google                             4837 non-null   float64
 7   fuente_OTROS                              4837 non-null   float64
 8   fuente_Organic Search                     4837 non-null   float64
 9   fuente_Reference                          4837 non-null   float64
 10  ult_actividad_Chat Conversation           483

## 4.1. Exportamos a pickle

In [58]:
ruta_proyecto = "../02_datos/03_Entrenamiento/"
ruta_df_tablon = ruta_proyecto + "04_df_tablon.pkl"

df_tablon.to_pickle(ruta_df_tablon)

# Avisamos al usuario
print("✅ Dataframe df_tablon exportado correctamente")

✅ Dataframe df_tablon exportado correctamente


## 4.2. Próximos pasos...

- En este caso, no se harán pre-selección de variables, ya que, al tratarse de **un proyecto que es "Segmentación" y no se va a realizar "Predicción"**. Aunque, dentro de la pre-selección tenemos el **componente de la correlación** lo que nos interesa mucho de cara a estudiar la colinealidad de las variables, ya que, **para hacer "Segmentación" NO es bueno incluir muchas variables que estén correlacionadas**.
- Tampoco se va a trabajar en el balanceo, ya que, al no hacer "Predicción" y además, la target no está desbalanceada.
- **Segmentación**, vamos a trabajar en la Segmentación y después al modelo predictivo.